## Aerial Vehicle Detection
The task mostly consists of two parts:

A. Auto-labeling data for training and evaluation

B. Train and object detection model to detect vehicles

Using free for commercial use Pexels clips without annotation, we need firstly create a pipeline for automatic pseudo ground truth labeling of single class vehicles objects. And then using such data train some lightweight vehicle detector and evaluate it.

In [2]:
import os
import re
import math
import shutil
import random
import json
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

I downloaded Pexel clips and create simple meta.json file for both train and eval clips, where added description, absolute path to video, and max vehicle size in pixels for video (I would describe this parameter later)

In [3]:
RAW_DATA_PATH = "./raw_data"
TRAIN_KEY = "train"
EVAL_KEY = "eval"
OUTPUT_DATA_PATH = "./data_sahi"
OUTPUT_GT_VIDEOS = "./data_sahi_gt_videos"

# Meta files
META_JSON = "meta.json"
VIDEO_PATH_KEY = "video_path"
DESCRIPTION_KEY = "description"
MAX_VEHICLE_PX_KEY = "max_vehicle_px"
LABELS_KEY = "labels"

# COCO class ids that we collapse into the single "vehicle" class.
# 2 car, 3 motorcycle, 5 bus, 7 truck. Bicycle (1) is excluded
VEHICLE_COCO_IDS = [2, 3, 5, 7]
BAND_NEAR = "0-200"
BAND_FAR = "200-400"
BAND_BEYOND = "400+"
BAND_NEAR_MAX_M = 200.0
BAND_FAR_MAX_M = 400.0

BAND_COLORS = {
    BAND_NEAR: (0, 200, 0),      # green   - near
    BAND_FAR: (0, 165, 255),     # orange  - far
    BAND_BEYOND: (128, 128, 128) # grey    - beyond 400 m
}


For auto-labeling I decided to use SOTA model Yolo26X.

Yolo26X model has highest accuracy but the biggest from all YOLO models. As training and evaluation data is usually a main KEY to efficient NN model, I decided to use the biggest and highest accuracy model. I also tried previous version YOLO 11X, but from my own tests YOLO 26X does better (precise) labeling. Additionnaly, I tried both YOLO 11X-OBB and YOLO 26X-OBB that trained for detecting rotated objects in aerial and satellite imagery, but in my own tests they miss more object. Need to be studied more...

In [4]:
# Teacher Detector Model 
AUTO_LABEL_MODEL = "yolo26x.pt"     # Model used
MODEL_CONF = 0.20                   # Default is 0.2, but better 0.20
MODEL_IMG_SIZE = 640                # 640 for original model, 1024 for OBB
                                    # Used for both SAHI and Model img size

# SAHI slicing
OVERLAP = 0.2                       # Probably can be 0.5, need to try
POSTPROCESS_TYPE = "GREEDYNMM"      # Variants ["GREEDYNMM", "NMM", "NMS", "NONE"]
                                    # Represents how SAHI merges overlapping slice detections.
MATCH_METRIC = "IOS"                # Variants ["IOU", "IOS"]. Intersection-over-union; Intersection-over-smaller)
MATCH_THRESHOLD = 0.5               # Lower - more aggressive merging of overlapping boxes
CLASS_AGNOSTIC = True               # Merges boxes across classes

# Post-filters
DEDUP_IOU=0.6                       # Final class-agnostic NMS over vehicle boxes. 1.0 disables. Need to be studied more
HFOV_DEG = 60.0
REF_LEN_M = 4.5                     # Reference length in meters for car (longest bbox size)

Load YOLO model with SAHI. As we focus on small objects on a video and used training and evaluation videos have high resolution, we cannot just pass frames into a YOLO model as it will automatically resize video to a training size (usually 640 for regular, and 1024 for OBB) and we will lost lots of details on image. Instead, we apply SAHI processing, where video is sliced for smaller overlapping regions of size expected into a model.

We also should consider desired output size of the model, that will be used on a drone camera. For my understanding, output training (student) model should not use SAHI or similar approaches and should be trained for original (probably resized) image sizes. But it can be discussed

In [15]:
device_str = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {AUTO_LABEL_MODEL} via SAHI on {device_str} with model input size {MODEL_IMG_SIZE}...")
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=AUTO_LABEL_MODEL,
    confidence_threshold=MODEL_CONF,
    device=device_str,
    image_size=MODEL_IMG_SIZE,
)
print("Model loaded!")

Loading yolo26x.pt via SAHI on cuda with model input size 640...
Model loaded!


In [8]:
def load_json(path):
    with open(path) as f:
        data = json.load(f)
        print(f"Loaded train_meta from {path}")
        return data

train_meta = load_json(os.path.join(RAW_DATA_PATH, TRAIN_KEY, META_JSON))
eval_meta = load_json(os.path.join(RAW_DATA_PATH, EVAL_KEY, META_JSON))

Loaded train_meta from ./raw_data\train\meta.json
Loaded train_meta from ./raw_data\eval\meta.json


I added max_vehicle_px that neededs to remove some glitches of big building that classified as vehicles. I manually set it and it needs some additional tunning probably. As an idea to automate it, it is better to add classes as buildings and if they overlapped with vehicles - we can delete them. Probably same can be appleid for small non-vehicles objects such as road signs.

In [5]:
def basename_any(p):
    """Basename that works for Windows (\\) or POSIX (/) paths on any OS."""
    return re.split(r"[\\/]", p.strip())[-1]


#TODO should be replaced by GeoCalib probably
def focal_px_from_fov(width_px, hfov_deg):
    """Focal length in pixels from horizontal FOV. f = (W/2) / tan(HFOV/2)."""
    return (width_px / 2.0) / math.tan(math.radians(hfov_deg) / 2.0)

#TODO should be replaced by GeoCalib probably
def estimate_distance_m(
    box_xyxy: tuple[float, float, float, float],
    focal_px: float,
    ref_len_m: float,
) -> float:
    """
    Estimate range to a vehicle from its apparent size.

    Pinhole model: a real object of length L projects to p pixels at distance D
    as  p = f * L / D , so  D = f * L / p.

    We take the LONGER side of the (axis-aligned) box as the apparent vehicle
    length p, and assume every vehicle is `ref_len_m` long. This is deliberately
    crude: it ignores vehicle-type variation, box-vs-true-length error when the
    car is not axis-aligned, and the fact that in top-down views box size is a
    footprint, not a range cue. It is good enough to *bin* objects into two
    coarse bands, which is all the task asks for. See README for caveats.
    """
    x1, y1, x2, y2 = box_xyxy
    long_side = max(x2 - x1, y2 - y1)
    if long_side <= 1e-6:
        return float("inf")
    return focal_px * ref_len_m / long_side


def read_video_meta(path):
    """fps / width / height / n_frames via OpenCV (no decode of all frames)."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {path}")
    meta = {
        "fps": float(cap.get(cv2.CAP_PROP_FPS)),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "n_frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return meta


def _dedup(cands, iou_thr):
    """Class-agnostic NMS over (box, conf): keep highest-conf, drop boxes that
    overlap a kept box at IoU >= iou_thr. Returns (kept, n_removed)."""
    kept, removed = [], 0
    for box, cf in sorted(cands, key=lambda t: -t[1]):
        if kept and iou_matrix([box], [b for b, _ in kept])[0].max() >= iou_thr:
            removed += 1
            continue
        kept.append((box, cf))
    return kept, removed


def band_for_distance(distance_m: float) -> str:
    if distance_m <= BAND_NEAR_MAX_M:
        return BAND_NEAR
    if distance_m <= BAND_FAR_MAX_M:
        return BAND_FAR
    return BAND_BEYOND


def iou_matrix(a, b):
    """Pairwise IoU between two lists of xyxy boxes -> (len(a), len(b)) array.
    Imported by detect.py (NMS), temporal.py (tracking) and metrics.py."""
    import numpy as np
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def xyxy_to_xywhn(
    box_xyxy: tuple[float, float, float, float], w: int, h: int
) -> list[float]:
    """Pixel xyxy -> normalized YOLO (xc, yc, w, h) in [0, 1]."""
    x1, y1, x2, y2 = box_xyxy
    xc = (x1 + x2) / 2.0 / w
    yc = (y1 + y2) / 2.0 / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return [xc, yc, bw, bh]

    
def label_video(video_path, description, detection_model, max_vehicle_px):
    """Run the SAHI detector over one video and build its rich GT record."""
    video_path = os.path.abspath(video_path)
    vmeta = read_video_meta(video_path)
    w, h, fps = vmeta["width"], vmeta["height"], vmeta["fps"]
    focal_px = focal_px_from_fov(w, HFOV_DEG)

    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_idx = 0
    n_large = n_dup = 0  # dropped counters (across the whole video)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # SAHI expects RGB arrays
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        result = get_sliced_prediction(
            frame_rgb,
            detection_model,
            slice_height=MODEL_IMG_SIZE,
            slice_width=MODEL_IMG_SIZE,
            overlap_height_ratio=OVERLAP,
            overlap_width_ratio=OVERLAP,
            postprocess_type=POSTPROCESS_TYPE,
            postprocess_match_metric=MATCH_METRIC,
            postprocess_match_threshold=MATCH_THRESHOLD,
            postprocess_class_agnostic=CLASS_AGNOSTIC,
            verbose=False,
        )

        # 1) collect vehicle candidates
        cands = []
        for obj in result.object_prediction_list:
            if obj.category.id in VEHICLE_COCO_IDS:
                box = [float(v) for v in obj.bbox.to_xyxy()]
                cands.append((box, float(obj.score.value)))

        # 2) drop boxes that are too large to be a vehicle (e.g. buildings)
        if max_vehicle_px is not None:
            filt = []
            for box, cf in cands:
                long_side = max(box[2] - box[0], box[3] - box[1])
                if long_side > max_vehicle_px:
                    n_large += 1
                    continue
                filt.append((box, cf))
            cands = filt

        # 3) drop overlapping duplicates of the same object
        if DEDUP_IOU < 1.0:
            cands, removed = _dedup(cands, DEDUP_IOU)
            n_dup += removed

        objects = []
        for box, cf in cands:
            dist = estimate_distance_m(box, focal_px, REF_LEN_M)
            objects.append(
                {
                    "bbox_xyxy": [round(v, 2) for v in box],
                    "bbox_xywhn": [round(v, 6) for v in xyxy_to_xywhn(box, w, h)],
                    "conf": round(float(cf), 4),
                    "class": "vehicle",
                    "distance_m": round(dist, 1) if dist != float("inf") else None,
                    "band": band_for_distance(dist),
                }
            )

        frames.append(
            {"frame": frame_idx, "time": round(frame_idx / fps, 4), "objects": objects}
        )

        frame_idx += 1

    cap.release()

    return {
        "video_path": video_path,
        "description": description,
        "fps": fps,
        "width": w,
        "height": h,
        "n_frames": vmeta["n_frames"],
        "detector": {
            "model": detection_model.model_path,
            "method": "SAHI",
            "conf": MODEL_CONF,
            "model_imgsz": MODEL_IMG_SIZE,
            "overlap": OVERLAP,
            "postprocess": {"type": POSTPROCESS_TYPE, "metric": MATCH_METRIC,
                            "threshold": MATCH_THRESHOLD, "class_agnostic": CLASS_AGNOSTIC},
            "vehicle_coco_ids": VEHICLE_COCO_IDS,
        },
        "filtering": {
            "max_vehicle_px": max_vehicle_px,
            "dropped_too_large": n_large,
            "dedup_iou": DEDUP_IOU,
            "dropped_duplicates": n_dup,
        },
        "distance": {
            "method": "pinhole, longer-box-side, single ref length",
            "hfov_deg": HFOV_DEG,
            "ref_vehicle_len_m": REF_LEN_M,
        },
        "frames": frames,
    }

def generate_gt_labels(meta, split):
    os.makedirs(OUTPUT_DATA_PATH, exist_ok=True)
    
    out_meta = []

    for entry in meta:
        video_path = entry[VIDEO_PATH_KEY]
        max_vehicle_px = entry.get(MAX_VEHICLE_PX_KEY, 10000)

        print(f"Meta: {meta}\n\t labeling {video_path} and max_vehicle_px {max_vehicle_px}")

        record = label_video(video_path, entry.get("description", ""), detection_model, max_vehicle_px) 

        ann_path = os.path.join(OUTPUT_DATA_PATH, LABELS_KEY, split, f"{os.path.splitext(basename_any(video_path))[0]}.json")
        os.makedirs(os.path.join(OUTPUT_DATA_PATH, LABELS_KEY, split), exist_ok=True)

        with open(ann_path, "w") as f:
            json.dump(record, f, indent=2)

        n_obj = sum(len(f["objects"]) for f in record["frames"])
        fl = record["filtering"]
        print(f"    -> {len(record['frames'])} frames, {n_obj} boxes "
              f"(dropped {fl['dropped_too_large']} too-large, "
              f"{fl['dropped_duplicates']} duplicates) -> {basename_any(video_path)}")

        out_meta.append(
            {
                "video_path": video_path,
                "annotations_path": ann_path,
                "description": entry.get("description", ""),
                "fps": record["fps"],
                "width": record["width"],
                "height": record["height"],
            }
        )

    meta_out_path = os.path.join(OUTPUT_DATA_PATH, f"{split}_meta.json")

    with open(meta_out_path, "w") as f:
        json.dump(out_meta, f, indent=2)
    
    print(f"\nWrote {meta_out_path}  ({len(out_meta)} videos)")

In [7]:
# Generate train meta
generate_gt_labels(train_meta, TRAIN_KEY)

# Generate eval meta
generate_gt_labels(eval_meta, EVAL_KEY)

Meta: [{'video_path': '.\\raw_data\\train\\8968356-hd_1920_1080_30fps.mp4', 'max_vehicle_px': 100, 'description': 'train A. Higheway interchange'}, {'video_path': '.\\raw_data\\train\\5382494-uhd_3840_2160_24fps.mp4', 'max_vehicle_px': 100, 'description': 'train B. Rural highway, sparse traffic'}, {'video_path': '.\\raw_data\\train\\8457857-uhd_3840_2160_24fps.mp4', 'max_vehicle_px': 1000, 'description': 'train C. Simple highway, top-down'}, {'video_path': '.\\raw_data\\train\\3405804-uhd_3840_2160_30fps.mp4', 'max_vehicle_px': 500, 'description': 'train D. Urban intersection'}]
	 labeling .\raw_data\train\8968356-hd_1920_1080_30fps.mp4 and max_vehicle_px 100
    -> 576 frames, 12411 boxes (dropped 29 too-large, 2 duplicates) -> 8968356-hd_1920_1080_30fps.mp4
Meta: [{'video_path': '.\\raw_data\\train\\8968356-hd_1920_1080_30fps.mp4', 'max_vehicle_px': 100, 'description': 'train A. Higheway interchange'}, {'video_path': '.\\raw_data\\train\\5382494-uhd_3840_2160_24fps.mp4', 'max_vehicle

Next cell will generate videos with Generated Ground-truth bboxes

In [8]:
def draw_box(img, obj, *, prefix=""):
    x1, y1, x2, y2 = (int(v) for v in obj["bbox_xyxy"])
    color = BAND_COLORS.get(obj["band"], (255, 255, 255))
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    d = obj.get("distance_m")
    label = f"{prefix}veh {d:.0f}m" if d is not None else f"{prefix}veh"
    cv2.putText(img, label, (x1, max(0, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)


def visualize_gt(annotation_path, out_path):
    ann = load_json(annotation_path)
    by_frame = {f["frame"]: f["objects"] for f in ann["frames"]}

    cap = cv2.VideoCapture(ann["video_path"])
    fps, w, h = ann["fps"], ann["width"], ann["height"]
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    frame_idx = 0
    while True:
        ok, img = cap.read()
        if not ok:
            break
        for obj in by_frame.get(frame_idx, []):
            draw_box(img, obj)
        # tiny legend
        cv2.putText(img, "green<200m  orange 200-400m  grey>400m",
                    (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        writer.write(img)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"Wrote {out_path}  ({frame_idx} frames)")

In [9]:
os.makedirs(OUTPUT_GT_VIDEOS, exist_ok=True)

# Draw Train
train_labels_dir = os.path.join(OUTPUT_DATA_PATH, LABELS_KEY, TRAIN_KEY)
for annotation in os.listdir(train_labels_dir):
    name = os.path.splitext(annotation)[0]  # file name without extension
    annotation_path = os.path.join(train_labels_dir, annotation)  # full path to annotation
    out_path = os.path.join(OUTPUT_GT_VIDEOS, f"{TRAIN_KEY}_{name}.mp4")
    visualize_gt(annotation_path, out_path)

# Draw Eval
eval_labels_dir = os.path.join(OUTPUT_DATA_PATH, LABELS_KEY, EVAL_KEY)
for annotation in os.listdir(os.path.join(OUTPUT_DATA_PATH, LABELS_KEY, EVAL_KEY)):
    name = os.path.splitext(annotation)[0]  # file name without extension
    annotation_path = os.path.join(eval_labels_dir, annotation)  # full path to annotation
    out_path = os.path.join(OUTPUT_GT_VIDEOS, f"{EVAL_KEY}_{name}.mp4")
    visualize_gt(annotation_path, out_path)

print(f"Output GT videos can be seen here: {OUTPUT_GT_VIDEOS}")

Loaded train_meta from ./data_sahi\labels\train\3405804-uhd_3840_2160_30fps.json
Wrote ./data_sahi_gt_videos\train_3405804-uhd_3840_2160_30fps.mp4  (734 frames)
Loaded train_meta from ./data_sahi\labels\train\5382494-uhd_3840_2160_24fps.json
Wrote ./data_sahi_gt_videos\train_5382494-uhd_3840_2160_24fps.mp4  (750 frames)
Loaded train_meta from ./data_sahi\labels\train\8457857-uhd_3840_2160_24fps.json
Wrote ./data_sahi_gt_videos\train_8457857-uhd_3840_2160_24fps.mp4  (405 frames)
Loaded train_meta from ./data_sahi\labels\train\8968356-hd_1920_1080_30fps.json
Wrote ./data_sahi_gt_videos\train_8968356-hd_1920_1080_30fps.mp4  (576 frames)
Loaded train_meta from ./data_sahi\labels\eval\12897527_1920_1080_30fps.json
Wrote ./data_sahi_gt_videos\eval_12897527_1920_1080_30fps.mp4  (1866 frames)
Loaded train_meta from ./data_sahi\labels\eval\13722965_2160_3840_30fps.json
Wrote ./data_sahi_gt_videos\eval_13722965_2160_3840_30fps.mp4  (918 frames)
Output GT videos can be seen here: ./data_sahi_gt_v

Training part is here:

In [6]:
TRAINING_MODEL = "yolo26n.pt"
WEIGHTS_OUT = "./weights/vehicle_yolo26n_640.pt"
EPOCHS = 80
BATCH_SIZE = 32
TILE_SIZE = MODEL_IMG_SIZE
TILE_OVERLAP = 0.2

FRAME_STRIDE = 5                           # Use every Nth labeled frame
FORCE_SLICE = False                        # Rebuild sliced dataset even if cache exists
SAVE_PERIOD = -1                           # Save checkpoints every N epochs; -1 means only best/last
NEG_FRAC = 0.1                             # Fraction of empty tiles to keep as negative samples
NMS_IOU = 0.55

# ============================================================
# Evaluation configuration
# ============================================================

INFER_CONF = 0.05                          # Low confidence threshold for PR-curve evaluation
OP_CONF = 0.25                             # Operating confidence threshold for final counting/inference
IOU_THR = 0.5                              # IoU threshold used for match/metric computation

In [9]:
prep_train_meta = load_json(os.path.join(OUTPUT_DATA_PATH, f"{TRAIN_KEY}_meta.json"))
prep_eval_meta = load_json(os.path.join(OUTPUT_DATA_PATH, f"{EVAL_KEY}_meta.json"))

Loaded train_meta from ./data_sahi\train_meta.json
Loaded train_meta from ./data_sahi\eval_meta.json


In [10]:
yolo_root = os.path.join(OUTPUT_DATA_PATH, "yolo_train")
data_yaml = os.path.join(yolo_root, "data.yaml")
manifest_path = os.path.join(yolo_root, "slice_manifest.json")

slice_params = {
    "tile": TILE_SIZE,
    "overlap": TILE_OVERLAP,
    "frame_stride": FRAME_STRIDE,
    "neg_frac": NEG_FRAC,
    "train_videos": [basename_any(e["video_path"]) for e in prep_train_meta],
    "val_videos": (
        [basename_any(e["video_path"]) for e in prep_eval_meta]
        if prep_eval_meta else None
    )
}

In [11]:
def _starts(total: int, tile: int, step: int) -> list[int]:
    if total <= tile:
        return [0]
    s = list(range(0, total - tile + 1, step))
    if s[-1] != total - tile:
        s.append(total - tile)
    return s


def make_tiles(w: int, h: int, tile: int, overlap: float) -> list[tuple[int, int, int, int]]:
    """Overlapping tile windows (x0,y0,x1,y1). tile<=0 -> single full frame."""
    if tile <= 0 or (w <= tile and h <= tile):
        return [(0, 0, w, h)]
    step = max(1, int(round(tile * (1.0 - overlap))))
    return [(x, y, min(x + tile, w), min(y + tile, h))
            for y in _starts(h, tile, step) for x in _starts(w, tile, step)]


def _clip_to_tile(box, tx0, ty0, tx1, ty1, min_vis):
    """Clip a full-frame xyxy box to a tile; keep if >= min_vis of it is inside.
    Returns tile-local xyxy or None."""
    ix1, iy1 = max(box[0], tx0), max(box[1], ty0)
    ix2, iy2 = min(box[2], tx1), min(box[3], ty1)
    if ix2 <= ix1 or iy2 <= iy1:
        return None
    orig = max(1e-6, (box[2] - box[0]) * (box[3] - box[1]))
    if (ix2 - ix1) * (iy2 - iy1) / orig < min_vis:
        return None  # only a sliver is in this tile -> skip (avoids cut-off labels)
    return [ix1 - tx0, iy1 - ty0, ix2 - tx0, iy2 - ty0]


def _slice_entries(entries, img_d, lab_d, *, tile, overlap, stride,
                   neg_frac, min_vis, rnd, force_split=None, val_dirs=None):
    """Slice a list of entries into img_d/lab_d. If force_split is None, each tile
    is routed to train or val per-tile.Returns (n_img, n_box, n_neg)."""
    n_img = n_box = n_neg = 0
    for entry in entries:
        ann = load_json(entry["annotations_path"])
        by_frame = {fr["frame"]: [o["bbox_xyxy"] for o in fr["objects"]] for fr in ann["frames"]}
        cap = cv2.VideoCapture(entry["video_path"])
        w, h = ann["width"], ann["height"]
        tiles = make_tiles(w, h, tile, overlap)
        stem = os.path.splitext(basename_any(entry["video_path"]))[0]
        for frame_idx in sorted(by_frame):
            if frame_idx % stride != 0:
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ok, img = cap.read()
            if not ok:
                continue
            boxes = by_frame[frame_idx]
            for ti, (tx0, ty0, tx1, ty1) in enumerate(tiles):
                tw, th = tx1 - tx0, ty1 - ty0
                local = [b for b in (_clip_to_tile(bx, tx0, ty0, tx1, ty1, min_vis)
                                     for bx in boxes) if b is not None]
                if not local and rnd.random() > neg_frac:
                    continue
                cur_img, cur_lab = img_d, lab_d
                name = f"{stem}_{frame_idx:06d}_t{ti:02d}"
                cv2.imwrite(os.path.join(cur_img, f"{name}.jpg"), img[ty0:ty1, tx0:tx1])
                lines = []
                for x1, y1, x2, y2 in local:
                    xc, yc = (x1 + x2) / 2 / tw, (y1 + y2) / 2 / th
                    bw, bh = (x2 - x1) / tw, (y2 - y1) / th
                    lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
                with open(os.path.join(cur_lab, f"{name}.txt"), "w", encoding="utf-8") as f:
                    f.write("\n".join(lines))
                n_img += 1
                n_box += len(local)
                n_neg += not local
        cap.release()
    return n_img, n_box, n_neg


def build_sliced_dataset(train_entries, yolo_root, *, val_entries=None,
                         tile=TILE_SIZE, overlap=0.2, stride=5,
                         neg_frac=0.1, min_vis=0.3, seed=0):
    """
    Slice into a YOLO dataset.
      - val_entries is None  -> legacy: random per-tile split of train_entries
                                (leaky across adjacent frames; use only as a default)
      - val_entries given    -> train_entries fully -> train, val_entries fully -> val
                                (video-level split; pass a held-out train video, or
                                 eval videos if you accept the leakage)
    """
    rnd = random.Random(seed)

    dirs = {
        s: (os.path.join(yolo_root, "images", s), os.path.join(yolo_root, "labels", s))
        for s in ("train", "val")
    }

    for img_d, lab_d in dirs.values():
        os.makedirs(img_d, exist_ok=True)
        os.makedirs(lab_d, exist_ok=True)

    if val_entries is None:
        n_img, n_box, n_neg = _slice_entries(
            train_entries, *dirs["train"], tile=tile, overlap=overlap, stride=stride,
            neg_frac=neg_frac, min_vis=min_vis, rnd=rnd, force_split=None,
            val_dirs=dirs["val"])
    else:
        nt = _slice_entries(train_entries, *dirs["train"], tile=tile, overlap=overlap,
                            stride=stride, neg_frac=neg_frac, min_vis=min_vis, rnd=rnd,
                            force_split="train")
        nv = _slice_entries(val_entries, *dirs["val"], tile=tile, overlap=overlap,
                            stride=stride, neg_frac=neg_frac, min_vis=min_vis, rnd=rnd,
                            force_split="val")
        n_img, n_box, n_neg = (nt[0] + nv[0], nt[1] + nv[1], nt[2] + nv[2])

    data_yaml = os.path.join(yolo_root, "data.yaml")
    with open(data_yaml, "w", encoding="utf-8") as f:
        f.write(
            f"path: {os.path.abspath(yolo_root)}\n"
            "train: images/train\n"
            "val: images/val\n"
            "names:\n"
            "  0: vehicle\n"
        )
    return {"images": n_img, "boxes": n_box, "negatives": n_neg, "data_yaml": str(data_yaml)}


In [12]:
cached_manifest = None
if not FORCE_SLICE and os.path.exists(data_yaml) and os.path.exists(manifest_path):
    try:
        m = load_json(manifest_path)
        if m.get("params") == slice_params:
            cached_manifest = m
    except Exception:
        cached_manifest = None

if cached_manifest is not None:
    stats = cached_manifest.get("stats", {})
    stats.setdefault("data_yaml", str(data_yaml))
    print(f"Reusing cached slices in {yolo_root} (params unchanged); "
          f"pass --force-slice to rebuild.")
else:
    print("Slicing into 640 tiles ...")
    if os.path.exists(yolo_root):
        shutil.rmtree(yolo_root)
    stats = build_sliced_dataset(
        prep_train_meta, yolo_root, val_entries=prep_eval_meta, tile=TILE_SIZE,
        overlap=TILE_OVERLAP, stride=FRAME_STRIDE, neg_frac=NEG_FRAC)
    with open(manifest_path, "w") as f:
        json.dump({"params": slice_params, "stats": stats}, f, indent=2)

print(f"  {stats.get('images', '?')} tiles, {stats.get('boxes', '?')} boxes, "
      f"{stats.get('negatives', '?')} negatives -> {stats['data_yaml']}")

Loaded train_meta from ./data_sahi\yolo_train\slice_manifest.json
Reusing cached slices in ./data_sahi\yolo_train (params unchanged); pass --force-slice to rebuild.
  9503 tiles, 40072 boxes, 1424 negatives -> ./data_sahi\yolo_train\data.yaml


In [15]:
print("Training ...")
model = YOLO(TRAINING_MODEL)
model.train(data=stats["data_yaml"],
            imgsz=MODEL_IMG_SIZE,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            device=device_str,
            single_cls=True,
            project="runs",
            name="vehicle_sahi640",
            mosaic=1.0,
            patience=20,
            save_period=SAVE_PERIOD)
best =  "./runs/detect/runs/vehicle_sahi640-4/weights/best.pt"

os.makedirs(os.path.dirname(WEIGHTS_OUT), exist_ok=True)
shutil.copy(best, WEIGHTS_OUT)
print(f"Saved trained weights -> {WEIGHTS_OUT}")
print("(ultralytics also kept runs/vehicle_sahi640/weights/last.pt and best.pt;"
      " per-epoch snapshots only if --save-period > 0)")

Training ...
New https://pypi.org/project/ultralytics/8.4.62 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.11.9 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./data_sahi\yolo_train\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, 

FileNotFoundError: [Errno 2] No such file or directory: './runs/vehicle_sahi640/weights/best.pt'

In [16]:
best = "./runs/detect/runs/vehicle_sahi640-4/weights/best.pt"

os.makedirs(os.path.dirname(WEIGHTS_OUT), exist_ok=True)
shutil.copy(best, WEIGHTS_OUT)
print(f"Saved trained weights -> {WEIGHTS_OUT}")
print("(ultralytics also kept runs/vehicle_sahi640-3/weights/last.pt and best.pt;"
      " per-epoch snapshots only if --save-period > 0)")

Saved trained weights -> ./weights/vehicle_yolo26n_640.pt
(ultralytics also kept runs/vehicle_sahi640-3/weights/last.pt and best.pt; per-epoch snapshots only if --save-period > 0)


Evaluate Part

In [13]:
SCORED_BANDS = [BAND_NEAR, BAND_FAR]
PRED_COLOR = (255, 0, 255)


def _iou(a, b) -> float:
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = ((a[2] - a[0]) * (a[3] - a[1]) +
          (b[2] - b[0]) * (b[3] - b[1]) - inter)
    return inter / ua if ua > 0 else 0.0


def nms(boxes, scores, iou_thr: float):
    """Greedy NMS. Returns indices of kept boxes (highest score first)."""
    if not boxes:
        return []
    boxes = np.asarray(boxes, dtype=float)
    scores = np.asarray(scores, dtype=float)
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(int(i))
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        w = np.clip(xx2 - xx1, 0, None)
        h = np.clip(yy2 - yy1, 0, None)
        inter = w * h
        iou = inter / np.maximum(areas[i] + areas[order[1:]] - inter, 1e-9)
        order = order[1:][iou <= iou_thr]
    return keep


def _match(gt: list, preds: list, iou_thr: float):
    """
    Greedy match (preds assumed sorted by descending conf).
    Returns (out, used):
      out  : list parallel to preds, each ("tp"|"fp"|"ignore", gt_index_or_-1)
             "ignore" = matched a GT whose band is not in SCORED_BANDS.
      used : list parallel to gt, True if that GT was matched.
    """
    used = [False] * len(gt)
    out = []
    for p in preds:
        best_iou, best_j = iou_thr, -1
        for j, g in enumerate(gt):
            if used[j]:
                continue
            i = _iou(p["xyxy"], g["xyxy"])
            if i >= best_iou:
                best_iou, best_j = i, j
        if best_j >= 0:
            used[best_j] = True
            kind = "tp" if gt[best_j]["band"] in SCORED_BANDS else "ignore"
            out.append((kind, best_j))
        else:
            out.append(("fp", -1))
    return out, used


def _average_precision(scores: list, tp: list, n_pos: int) -> float:
    """All-points (VOC2010+) AP."""
    if n_pos == 0 or not scores:
        return 0.0
    order = np.argsort(-np.asarray(scores, dtype=float))
    tp_arr = np.asarray(tp, dtype=float)[order]
    fp_arr = 1.0 - tp_arr
    tp_cum = np.cumsum(tp_arr)
    fp_cum = np.cumsum(fp_arr)
    recall = tp_cum / n_pos
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))


# ============================ METRICS ============================
def gt_frames_from_json(ann: dict) -> dict:
    """Rich per-video GT JSON -> {frame_idx: [{"xyxy","band"}]} for metrics."""
    return {fr["frame"]: [{"xyxy": o["bbox_xyxy"], "band": o["band"]} for o in fr["objects"]]
            for fr in ann["frames"]}


def compute_metrics_multi(items: list, iou_thr: float = 0.5, conf_thr: float = 0.25) -> dict:
    """
    items: list of (gt_by_frame, pred_by_frame, fps) - one tuple per video.
    Aggregates across videos with each video's OWN fps (so 24/30 fps mix is
    handled correctly for false-alarms/min). TTFD per band is the mean of each
    video's first-detection time (videos with no TP in a band are skipped).
    """
    ap_scores, ap_tp, n_pos = [], [], 0
    TP = {b: 0 for b in SCORED_BANDS}
    FP = {b: 0 for b in SCORED_BANDS}
    FN = {b: 0 for b in SCORED_BANDS}
    ttfd_list = {b: [] for b in SCORED_BANDS}
    total_duration = 0.0
    n_eval = 0

    for gt_by_frame, pred_by_frame, fps in items:
        frames = sorted(set(gt_by_frame) | set(pred_by_frame))
        n_eval += len(frames)
        total_duration += len(frames) / fps if fps > 0 else 0.0
        first_tp = {b: None for b in SCORED_BANDS}

        for f in frames:
            gt = gt_by_frame.get(f, [])
            n_pos += sum(1 for g in gt if g["band"] in SCORED_BANDS)
            # Pass A (AP): all predictions
            preds_all = sorted(pred_by_frame.get(f, []), key=lambda p: -p["conf"])
            outA, _ = _match(gt, preds_all, iou_thr)
            for p, (kind, _j) in zip(preds_all, outA):
                if kind == "ignore":
                    continue
                ap_scores.append(p["conf"])
                ap_tp.append(1 if kind == "tp" else 0)
            # Pass B (counts/TTFD): predictions at the operating threshold
            preds = sorted((p for p in pred_by_frame.get(f, []) if p["conf"] >= conf_thr),
                           key=lambda p: -p["conf"])
            outB, used = _match(gt, preds, iou_thr)
            for p, (kind, j) in zip(preds, outB):
                if kind == "tp":
                    band = gt[j]["band"]
                    TP[band] += 1
                    if first_tp[band] is None:
                        first_tp[band] = f / fps if fps > 0 else 0.0
                elif kind == "fp" and p["band"] in SCORED_BANDS:
                    FP[p["band"]] += 1
            for j, g in enumerate(gt):
                if not used[j] and g["band"] in SCORED_BANDS:
                    FN[g["band"]] += 1

        for b in SCORED_BANDS:
            if first_tp[b] is not None:
                ttfd_list[b].append(first_tp[b])

    ap50 = _average_precision(ap_scores, ap_tp, n_pos)
    out = {"mAP@0.5_both_bands": round(ap50, 4), "n_eval_frames": n_eval,
           "n_videos": len(items), "bands": {}}
    for b in SCORED_BANDS:
        tp, fp, fn = TP[b], FP[b], FN[b]
        ttfd = round(sum(ttfd_list[b]) / len(ttfd_list[b]), 3) if ttfd_list[b] else None
        out["bands"][b] = {
            "TP": tp, "FP": fp, "FN": fn,
            "detection_rate": round(tp / (tp + fn), 4) if (tp + fn) else None,
            "precision": round(tp / (tp + fp), 4) if (tp + fp) else None,
            "false_alarms_per_min": round(fp / total_duration * 60, 3) if total_duration else None,
            "time_to_first_detection_s": ttfd,
        }
    return out


def format_table(title: str, res: dict) -> str:
    """Render the required metric table (rows = metrics, cols = bands)."""
    b = res["bands"]
    n, f = BAND_NEAR, BAND_FAR

    def cell(band, key, fmt="{:.3f}"):
        v = b[band][key]
        return "n/a" if v is None else fmt.format(v)

    rows = [
        ("Detection rate  TP/(TP+FN)", "detection_rate", "{:.3f}"),
        ("Precision       TP/(TP+FP)", "precision", "{:.3f}"),
        ("False alarms / min", "false_alarms_per_min", "{:.2f}"),
        ("Time to first detection (s)", "time_to_first_detection_s", "{:.2f}"),
    ]
    w = 30
    lines = [f"\n=== {title} ===",
             f"{'Metric':<{w}}{n:>12}{f:>12}",
             "-" * (w + 24)]
    for label, key, fmt in rows:
        lines.append(f"{label:<{w}}{cell(n, key, fmt):>12}{cell(f, key, fmt):>12}")
    lines.append("-" * (w + 24))
    lines.append(f"mAP@0.5 across both bands: {res['mAP@0.5_both_bands']:.4f}   "
                 f"(videos={res['n_videos']}, frames={res['n_eval_frames']})")
    counts = "  ".join(f"{bn}: TP={b[bn]['TP']} FP={b[bn]['FP']} FN={b[bn]['FN']}"
                       for bn in (n, f))
    lines.append("counts -> " + counts)
    return "\n".join(lines)


# ============================ SAHI INFERENCE ============================
def predict_video_sahi(model, video_path: str, *, width: int, hfov_deg: float,
                       ref_len_m: float, tile=TILE_SIZE, overlap=0.2, conf=0.05,
                       device="0", nms_iou=NMS_IOU, height=None):
    """
    Sliced inference: slice each frame into native-res tiles, run the student on
    each, map boxes back, NMS-merge. Returns ({frame: [{xyxy,conf,band}]}, n_frames).
    """
    focal_px = focal_px_from_fov(width, hfov_deg)
    cap = cv2.VideoCapture(str(video_path))
    h = height or int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    tiles = make_tiles(width, h, tile, overlap)
    preds, idx = {}, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        boxes, scores = [], []
        for (x0, y0, x1, y1) in tiles:
            r = model.predict(frame[y0:y1, x0:x1], imgsz=tile, conf=conf,
                              device=device, verbose=False)[0]
            if r.boxes is None or len(r.boxes) == 0:
                continue
            xy = r.boxes.xyxy.cpu().numpy()
            cf = r.boxes.conf.cpu().numpy()
            for bx, c in zip(xy, cf):
                boxes.append([float(bx[0]) + x0, float(bx[1]) + y0,
                              float(bx[2]) + x0, float(bx[3]) + y0])
                scores.append(float(c))
        keep = nms(boxes, scores, nms_iou)
        items = []
        for i in keep:
            d = estimate_distance_m(boxes[i], focal_px, ref_len_m)
            items.append({"xyxy": boxes[i], "conf": scores[i], "band": band_for_distance(d)})
        preds[idx] = items
        idx += 1
    cap.release()
    return preds, idx


# ============================ VISUALIZATION ============================
def _box(img, xyxy, color, thickness, label):
    x1, y1, x2, y2 = (int(v) for v in xyxy)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    if label:
        cv2.putText(img, label, (x1, max(0, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)


def visualize_video(video_path: str, ann: dict, preds: dict, *, out_dir: str,
                    draw_conf: float = OP_CONF):
    """
    Write <stem>_gt_vs_pred.mp4 (GT + preds) and <stem>_pred_only.mp4 (preds).
    Reuses the already-computed `preds` so it does NOT re-run inference.
    Returns (path_both, path_pred_only).
    """
    w, h, fps = ann["width"], ann["height"], ann["fps"]
    gt_by_frame = {fr["frame"]: fr["objects"] for fr in ann["frames"]}

    os.makedirs(out_dir, exist_ok=True)
    stem = os.path.splitext(os.path.basename(video_path))[0]
    path_both = os.path.join(out_dir, f"{stem}_gt_vs_pred.mp4")
    path_pred = os.path.join(out_dir, f"{stem}_pred_only.mp4")

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    vp_both = cv2.VideoWriter(path_both, fourcc, fps, (w, h))
    vp_pred = cv2.VideoWriter(path_pred, fourcc, fps, (w, h))

    cap = cv2.VideoCapture(video_path)
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        both = frame.copy()
        pred_only = frame.copy()

        # ground truth (band-colored, solid) on the overlay only
        for o in gt_by_frame.get(idx, []):
            _box(both, o["bbox_xyxy"], BAND_COLORS.get(o["band"], (255, 255, 255)), 2, "GT")
        # predictions: magenta on overlay, band-colored on pred-only
        for p in preds.get(idx, []):
            if p["conf"] < draw_conf:
                continue
            _box(both, p["xyxy"], PRED_COLOR, 1, f"{p['conf']:.2f}")
            _box(pred_only, p["xyxy"], BAND_COLORS.get(p["band"], (255, 255, 255)), 2,
                 f"veh {p['conf']:.2f}")

        cv2.putText(both, "GT: green<200m / orange 200-400m   PRED: magenta",
                    (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
        vp_both.write(both)
        vp_pred.write(pred_only)
        idx += 1

    cap.release()
    vp_both.release()
    vp_pred.release()
    return path_both, path_pred


# ============================ SPLIT EVALUATION ============================
def build_meta(labels_dir: str) -> list:
    """Scan a directory for *.json annotation files -> meta entries."""
    entries = []
    if not labels_dir or not os.path.isdir(labels_dir):
        return entries
    for name in sorted(os.listdir(labels_dir)):
        if name.lower().endswith(".json"):
            entries.append({"annotations_path": os.path.join(labels_dir, name)})
    return entries


def evaluate_split(weights, meta_entries, *, tile, overlap, conf, device, nms_iou,
                   eval_conf_thr, iou_thr, out_dir, draw_conf=OP_CONF, write_videos=True):
    """
    SAHI-infer every video in a split. For each video: compute metrics AND
    (optionally) write the two visualization videos. Inference runs once per
    video and the predictions are reused for both metrics and drawing.
    Returns (table_input_items, per_video).
    """
    model = YOLO(str(weights))
    items, per_video = [], []
    for entry in meta_entries:
        ann = load_json(entry["annotations_path"])
        video_path = ann["video_path"]
        preds, _ = predict_video_sahi(
            model, video_path, width=ann["width"], height=ann["height"],
            hfov_deg=ann["distance"]["hfov_deg"], ref_len_m=ann["distance"]["ref_vehicle_len_m"],
            tile=tile, overlap=overlap, conf=conf, device=device, nms_iou=nms_iou)
        gt = gt_frames_from_json(ann)
        items.append((gt, preds, ann["fps"]))
        res = compute_metrics_multi([(gt, preds, ann["fps"])], iou_thr, eval_conf_thr)
        desc = entry.get("description", os.path.splitext(os.path.basename(video_path))[0])
        per_video.append((desc, res))

        if write_videos:
            p_both, p_pred = visualize_video(video_path, ann, preds,
                                             out_dir=out_dir, draw_conf=draw_conf)
            print(f"  wrote: {p_both}\n         {p_pred}")
    return items, per_video

In [16]:
OUT_DIR_VIZ = "out_train_viz"

common_kw = dict(tile=TILE_SIZE, overlap=TILE_OVERLAP, conf=INFER_CONF,
                 device=device_str, nms_iou=NMS_IOU,
                 eval_conf_thr=OP_CONF, iou_thr=IOU_THR,
                 draw_conf=OP_CONF, write_videos=True)

for split_name, meta in (("TRAIN", prep_train_meta), ("EVAL (held-out)", prep_eval_meta)):
    split_out = os.path.join(OUT_DIR_VIZ, split_name.split()[0].lower())  # out/train, out/eval
    items, per_video = evaluate_split(WEIGHTS_OUT, meta, out_dir=split_out, **common_kw)
    for desc, res in per_video:
        print(format_table(f"{split_name} :: {desc}", res))
    agg = compute_metrics_multi(items, IOU_THR, OP_CONF)
    print(format_table(f"{split_name} :: ALL VIDEOS (aggregate)", agg))

Loaded train_meta from ./data_sahi\labels\train\8968356-hd_1920_1080_30fps.json
  wrote: out_train_viz\train\8968356-hd_1920_1080_30fps_gt_vs_pred.mp4
         out_train_viz\train\8968356-hd_1920_1080_30fps_pred_only.mp4
Loaded train_meta from ./data_sahi\labels\train\5382494-uhd_3840_2160_24fps.json
  wrote: out_train_viz\train\5382494-uhd_3840_2160_24fps_gt_vs_pred.mp4
         out_train_viz\train\5382494-uhd_3840_2160_24fps_pred_only.mp4
Loaded train_meta from ./data_sahi\labels\train\8457857-uhd_3840_2160_24fps.json
  wrote: out_train_viz\train\8457857-uhd_3840_2160_24fps_gt_vs_pred.mp4
         out_train_viz\train\8457857-uhd_3840_2160_24fps_pred_only.mp4
Loaded train_meta from ./data_sahi\labels\train\3405804-uhd_3840_2160_30fps.json
  wrote: out_train_viz\train\3405804-uhd_3840_2160_30fps_gt_vs_pred.mp4
         out_train_viz\train\3405804-uhd_3840_2160_30fps_pred_only.mp4

=== TRAIN :: train A. Higheway interchange ===
Metric                               0-200     200-400
----

Interesting, that added second eval clip `12897527_1920_1080_30fps.mp4` works similarly as training video `8968356-hd_1920_1080_30fps.mp4`. At the begging of `12897527_1920_1080_30fps.mp4`, when cars are placed orthogonally, detector works only on closest part of the road, but drone change perspective and cars start moving less vertically, detector works better. Same for Auto-labelling YOLO 26X model for `8968356-hd_1920_1080_30fps.mp4`, only cars moving not vertically detected efficiently. It is probably should be fixed at labeling stage and then trained model will perform better.